**Github** : https://github.com/OzgurYldrm/AI-ML-Course     
**Youtube** : https://www.youtube.com/@F%C3%BCt%C3%BCrist_AIntelligence

In [4]:
import numpy as np

# PreProcess

In [8]:
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/ozgur/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/ozgur/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
text = "Hello there! This is an example, showing basic text preprocessing."

text_lower = text.lower()
print("Lowercased:", text_lower)

text_no_punct = text_lower.translate(str.maketrans('', '', string.punctuation))
print("No Punctuation:", text_no_punct)

tokens = word_tokenize(text_no_punct)
stop_words = set(stopwords.words('english'))
tokens_cleaned = [word for word in tokens if word not in stop_words]

print("After Stopword Removal:", tokens_cleaned)

Lowercased: hello there! this is an example, showing basic text preprocessing.
No Punctuation: hello there this is an example showing basic text preprocessing
After Stopword Removal: ['hello', 'example', 'showing', 'basic', 'text', 'preprocessing']


In [18]:
import re
text = "I have 2 apples and 15 oranges in 2024."
text_no_digits = re.sub(r'\d+', '', text)
print(text_no_digits)

I have  apples and  oranges in .


# Stemming - Lemmatization

In [ ]:
from stemming.lovins import stem
print(stem("Artificial"))

yapa


In [20]:
from nltk.stem import PorterStemmer
porter_stemmer = PorterStemmer()
porter_stemmer.stem("Artificial")

'artifici'

In [22]:
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')  # WordNet için ek dil desteği

[nltk_data] Downloading package wordnet to /home/ozgur/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ozgur/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [30]:
lemmatizer = WordNetLemmatizer()

words = ["running", "ran", "better", "cars"]

for w in words:
    print(f"{w} --> {lemmatizer.lemmatize(w, pos='v')}")  # pos='v' ile fiil olarak lemmatize

running --> run
ran --> run
better --> better
cars --> cars


# Tokenization

In [31]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /home/ozgur/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [32]:
from nltk.tokenize import word_tokenize,sent_tokenize

In [34]:
text = "May the Force be with you. Remember, the Force will be around you always."
print(word_tokenize(text))
print(sent_tokenize(text))

['May', 'the', 'Force', 'be', 'with', 'you', '.', 'Remember', ',', 'the', 'Force', 'will', 'be', 'around', 'you', 'always', '.']
['May the Force be with you.', 'Remember, the Force will be around you always.']


In [36]:
print(list("May the Force be with you"))

['M', 'a', 'y', ' ', 't', 'h', 'e', ' ', 'F', 'o', 'r', 'c', 'e', ' ', 'b', 'e', ' ', 'w', 'i', 't', 'h', ' ', 'y', 'o', 'u']


In [38]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

bert tokenizer eğer kelime vocabulary dictionary içerisinde varsa kelimenin tamamını bir token olarak döndürür 

In [40]:
text = "May the Force be with you."
tokens = tokenizer.tokenize(text)
print(tokens)

text = "antidisestablishmentarianis"
tokens = tokenizer.tokenize(text)
print(tokens)

['may', 'the', 'force', 'be', 'with', 'you', '.']
['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##is']


# Byte-Pair Encoding

In [28]:
from collections import Counter
from datasets import load_dataset

def collect_corpus(dataset, max_docs=5_000):
    corpus = []
    for i, row in enumerate(dataset):
        if i >= max_docs:
            break
        text = row.get("text", "")
        if text:
            corpus.append(text)
    return corpus

def build_initial_vocab(corpus):
    """
    return: Counter({ tuple(token): freq })
    """
    Vocab = Counter()

    for doc in corpus:
        for word in doc.strip().split():
            tokens = tuple(word) + ("</w>",)
            Vocab[tokens] += 1
    return Vocab

def get_best_pair(Vocab):
    pair_counter = Counter()

    for word, freq in Vocab.items():
        for i in range(len(word) - 1):
            pair_counter[(word[i], word[i + 1])] += freq

    if not pair_counter:
        return None

    return pair_counter.most_common(1)[0][0]

def merge_pair(Vocab, pair):
    a, b = pair
    merged = a + b

    new_Vocab = Counter()

    for word, freq in Vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
                new_word.append(merged)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_Vocab[tuple(new_word)] += freq

    return new_Vocab

In [16]:
max_tokens = 1000
max_docs = 100

ds = load_dataset("wikimedia/wikipedia","20231101.tr",split="train") # Dataset

corpus = collect_corpus(ds, max_docs=max_docs)

In [19]:
Vocab = build_initial_vocab(corpus) # Initial vocab

In [21]:
token_set = set() # Global token set
for word in Vocab:
    token_set.update(word)

In [29]:
# BPE training loop
for step in range(max_tokens):
    best_pair = get_best_pair(Vocab)
    if best_pair is None:
        break

    merged_token = best_pair[0] + best_pair[1]
    token_set.add(merged_token)

    Vocab = merge_pair(Vocab, best_pair)

    if step % 10 == 0:
        print(f"Step {step}: merged {best_pair} -> {merged_token}")

Step 0: merged ('n', '</w>') -> n</w>
Step 10: merged ('.', '</w>') -> .</w>
Step 20: merged ('e', 't') -> et
Step 30: merged ('d', 'e') -> de
Step 40: merged ('e', 'm') -> em
Step 50: merged ('a', 'd') -> ad
Step 60: merged ('a', 'z') -> az
Step 70: merged ('a', 'p') -> ap
Step 80: merged ('u', 'm') -> um
Step 90: merged ('in', 'de</w>') -> inde</w>
Step 100: merged ('ı', 'k') -> ık
Step 110: merged ('u', 'n</w>') -> un</w>
Step 120: merged ('l', 'a</w>') -> la</w>
Step 130: merged ('ler', 'i</w>') -> leri</w>
Step 140: merged ('iş', 't') -> işt
Step 150: merged ('u', 's') -> us
Step 160: merged ('m', 'ış') -> mış
Step 170: merged ('i', 'b') -> ib
Step 180: merged ('ı', 'y') -> ıy
Step 190: merged ('o', 'n</w>') -> on</w>
Step 200: merged ('B', 'u</w>') -> Bu</w>
Step 210: merged ('e', 'ğ') -> eğ
Step 220: merged ('ay', 'a</w>') -> aya</w>
Step 230: merged ('t', 'ır.</w>') -> tır.</w>
Step 240: merged ('Türk', 'iy') -> Türkiy
Step 250: merged ('c', 'a</w>') -> ca</w>
Step 260: merged 

In [30]:
def bpe_encode_word(word, token_set):
    """
    word: str
    token_set: set(str)
    return: list[str]
    """
    tokens = list(word) + ["</w>"]

    i = 0
    output = []

    while i < len(tokens):
        matched = None

        # en uzun tokenu bulmaya çalış
        for j in range(len(tokens), i, -1):
            candidate = "".join(tokens[i:j])
            if candidate in token_set:
                matched = candidate
                i = j
                break

        if matched is None: # fallback
            matched = tokens[i]
            i += 1

        output.append(matched)

    return output

In [31]:
print(bpe_encode_word("yapay", token_set))
print(bpe_encode_word("öğrenme", token_set))
print(bpe_encode_word("Geliştirme", token_set))
print(bpe_encode_word("ve", token_set))

['yap', 'ay</w>']
['öğren', 'me</w>']
['G', 'el', 'iştir', 'me</w>']
['ve</w>']


In [33]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=1000,
    min_frequency=2,
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
)


tokenizer.train_from_iterator(corpus, trainer=trainer)

output = tokenizer.encode("fütürist")

print("Tokenlar:", output.tokens)
print("ID'ler:", output.ids)




Tokenlar: ['f', 'ü', 'tür', 'ist']
ID'ler: [73, 131, 562, 431]


# Word Vectors

In [68]:
import gensim.downloader as api

model = api.load("glove-wiki-gigaword-100") #128 MB

In [71]:
model["king"]

array([-0.32307 , -0.87616 ,  0.21977 ,  0.25268 ,  0.22976 ,  0.7388  ,
       -0.37954 , -0.35307 , -0.84369 , -1.1113  , -0.30266 ,  0.33178 ,
       -0.25113 ,  0.30448 , -0.077491, -0.89815 ,  0.092496, -1.1407  ,
       -0.58324 ,  0.66869 , -0.23122 , -0.95855 ,  0.28262 , -0.078848,
        0.75315 ,  0.26584 ,  0.3422  , -0.33949 ,  0.95608 ,  0.065641,
        0.45747 ,  0.39835 ,  0.57965 ,  0.39267 , -0.21851 ,  0.58795 ,
       -0.55999 ,  0.63368 , -0.043983, -0.68731 , -0.37841 ,  0.38026 ,
        0.61641 , -0.88269 , -0.12346 , -0.37928 , -0.38318 ,  0.23868 ,
        0.6685  , -0.43321 , -0.11065 ,  0.081723,  1.1569  ,  0.78958 ,
       -0.21223 , -2.3211  , -0.67806 ,  0.44561 ,  0.65707 ,  0.1045  ,
        0.46217 ,  0.19912 ,  0.25802 ,  0.057194,  0.53443 , -0.43133 ,
       -0.34311 ,  0.59789 , -0.58417 ,  0.068995,  0.23944 , -0.85181 ,
        0.30379 , -0.34177 , -0.25746 , -0.031101, -0.16285 ,  0.45169 ,
       -0.91627 ,  0.64521 ,  0.73281 , -0.22752 , 

In [8]:
print(model["king"].shape)

(100,)


In [74]:
print(np.dot(model["apple"],model["chair"]))
print(np.dot(model["apple"],model["banana"]))

3.4636424
16.466734


In [76]:
print(np.dot(model["magic"],model["merlin"]))
print(np.dot(model["magic"],model["harry"]))

8.64325
11.184023


In [78]:
vector = model["king"] - model["man"] + model["woman"]
nearest = model.similar_by_vector(vector, topn=10)
for i in nearest:
    print(i)

('king', 0.8551837205886841)
('queen', 0.783441424369812)
('monarch', 0.6933802366256714)
('throne', 0.6833109855651855)
('daughter', 0.6809081435203552)
('prince', 0.6713141798973083)
('princess', 0.664408266544342)
('mother', 0.6579325795173645)
('elizabeth', 0.6563301086425781)
('father', 0.6392418742179871)


# Bag-of-Words

In [79]:
class BagOfWords:
    def __init__(self):
        self.vocab = {}
        self.vocab_size = 0

    def tokenize(self, text):
        return text.lower().split()

    def fit(self, documents):
        """
        Train: Create a vocabulary containing teh words in the documents.
        """
        for doc in documents:
            for word in self.tokenize(doc):
                if word not in self.vocab:
                    self.vocab[word] = self.vocab_size
                    self.vocab_size += 1

    def transform(self, documents):
        """
        Vectorize: Create vectors of given documents regarding the vocab trained on corpus.
        """
        import numpy as np
        vectors = np.zeros((len(documents), self.vocab_size), dtype=int)

        for i, doc in enumerate(documents):
            for word in self.tokenize(doc):
                if word in self.vocab:       # ignore <UNK>
                    idx = self.vocab[word]
                    vectors[i][idx] += 1

        return vectors

    def fit_transform(self, documents):
        """
        Combines fit and transform
        """
        self.fit(documents)
        return self.transform(documents)


In [83]:
sentences = [
    "Kedi oturdu",
    "Köpek koştu"
]

bow_model = BagOfWords()
bow_model.fit(sentences)

print(bow_model.vocab)
print(bow_model.transform(["Kedi ayağa kalktı köpek oturdu","Kedi köpek oturdu"]))

{'kedi': 0, 'oturdu': 1, 'köpek': 2, 'koştu': 3}
[[1 1 1 0]
 [1 1 1 0]]


In [84]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "Kedi oturdu",
    "Köpek hızlı koştu",
    "Kedi tekrar koştu"
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)

print("Vocabulary:", vectorizer.vocabulary_)
print(X.toarray())

print(vectorizer.transform(["Köpek kedi oturdu"]).toarray())

Vocabulary: {'kedi': 1, 'oturdu': 4, 'köpek': 3, 'hızlı': 0, 'koştu': 2, 'tekrar': 5}
[[0 1 0 0 1 0]
 [1 0 1 1 0 0]
 [0 1 1 0 0 1]]
[[0 1 0 1 1 0]]


# Term Frequency

In [1]:
s1 = "The cat sat on the mat"
s2 = "The dog played in the park"
s3 = "We have the cat and the dog"

In [2]:
class Term_Frequency:
    def __init__(self):
        self.vocab = {}
        self.vocab_size = 0

    def tokenize(self, text):
        return text.lower().split()

    def fit(self, documents):
        """
        Train: Create a vocabulary containing the words in the documents.
        """
        for doc in documents:
            for word in self.tokenize(doc):
                if word not in self.vocab:
                    self.vocab[word] = self.vocab_size
                    self.vocab_size += 1

    def transform(self, documents):
        vectors = np.zeros((len(documents), self.vocab_size), dtype=float)
        for i, doc in enumerate(documents):
            for word in self.tokenize(doc):
                if word in self.vocab:
                    vectors[i][self.vocab[word]] += 1
            row_sum = vectors[i].sum()
            if row_sum > 0:
                vectors[i] /= row_sum
        return vectors

    def fit_transform(self, documents):
        """
        Combines fit and transform
        """
        self.fit(documents)
        return self.transform(documents)


In [ ]:
s1 = "The cat sat on the mat"

In [7]:
tf = Term_Frequency()
tf.fit([s1,s2,s3])
print(tf.vocab)
tf.transform([s1])

{'the': 0, 'cat': 1, 'sat': 2, 'on': 3, 'mat': 4, 'dog': 5, 'played': 6, 'in': 7, 'park': 8, 'we': 9, 'have': 10, 'and': 11}


array([[0.33333333, 0.16666667, 0.16666667, 0.16666667, 0.16666667,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ]])

# Word2Vec

In [ ]:
import gensim.downloader as api
#model = api.load("word2vec-google-news-300") #1.6 GB

In [4]:
vector = model["Paris"] - model["France"] + model["Turkey"]
nearest = model.similar_by_vector(vector, topn=10)
for i in nearest:
    print(i)

('Ankara', 0.735222339630127)
('Istanbul', 0.7234363555908203)
('Turkey', 0.7217690944671631)
('İstanbul', 0.6729998588562012)
('Baku', 0.6014183163642883)
('Turkish', 0.5942798852920532)
('İzmir', 0.5655789375305176)
('Istanbul_Turkey', 0.5597474575042725)
('Gül', 0.5528533458709717)
('Amman', 0.5425913333892822)


# GloVe

In [10]:
import gensim.downloader as api

model = api.load("glove-wiki-gigaword-50")

print("Vocab size:", len(model))

vec = model["king"]
print(vec.shape)

print(model.most_similar("king", topn=5))

print(model.similarity("king", "queen"))

Vocab size: 400000
(50,)
[('prince', 0.8236179351806641), ('queen', 0.7839043140411377), ('ii', 0.7746230363845825), ('emperor', 0.7736247777938843), ('son', 0.766719400882721)]
0.7839043


# FastText

In [11]:
from gensim.models import FastText

sentences = [
    ["ben", "kitap", "okuyorum"],
    ["sen", "kitap", "okudun"],
    ["o", "film", "izliyor"],
    ["biz", "film", "izledik"],
    ["kitaplar", "çok", "güzeldir"],
    ["filmler", "bazen", "uzun", "olur"]
]

model = FastText(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    min_n=3,
    max_n=5,
    epochs=20
)

print(model.wv["kitap"])

print(model.wv["kitapçı"])

[-2.3720525e-03  1.5221797e-03 -4.9279830e-03  7.2263363e-05
 -1.1105894e-03 -6.3759089e-03 -2.4029841e-03 -2.0061822e-03
  2.4048989e-03 -9.3651708e-04 -5.5647874e-03  2.7459215e-03
  1.4254766e-03 -5.8893310e-03  1.1005453e-03 -2.4094379e-03
  4.0004545e-04 -4.3571177e-03  2.9387302e-03  1.4815156e-03
  1.6305200e-03  3.6503684e-03 -1.4371704e-03  2.7655922e-03
 -6.5149146e-04 -3.4608119e-03  3.1368898e-03  1.1522148e-03
 -4.8418916e-04 -4.1580531e-03  5.0680424e-05 -2.9723410e-04
  2.0434035e-03 -2.9944738e-03 -2.6894172e-03 -9.9067402e-04
 -3.9369245e-03 -6.8339333e-04 -2.9537645e-03  4.5423941e-03
 -5.5595945e-05  6.7804451e-04  7.8185461e-05 -3.5588641e-03
  4.3565705e-03 -2.3019228e-04  5.4464568e-03  7.1593240e-04
 -2.1485754e-03  1.1253107e-03]
[-0.00090799 -0.00203229 -0.00146853 -0.00463013 -0.00235172 -0.00287288
 -0.00120457  0.00509267  0.0030834  -0.00141147 -0.00136892  0.00253896
  0.00034383 -0.00463363  0.00382988  0.00099362  0.00604708 -0.00268112
  0.00283934  0.0

In [ ]:
import gensim.downloader as api
#model = api.load("fasttext-wiki-news-subwords-300")  # ~7GB

# Greedy Search - Beam Search

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

In [10]:
text = "Alan Turing is"
input_ids = tokenizer.encode(text, return_tensors="pt")

In [12]:
predicted_tokens = model.generate(input_ids, max_new_tokens=8 ,pad_token_id=tokenizer.eos_token_id)
predicted_text = tokenizer.decode(predicted_tokens[0])
print(predicted_text)

predicted_tokens = model.generate(input_ids, num_beams=5, max_new_tokens=8 ,pad_token_id=tokenizer.eos_token_id)
predicted_text = tokenizer.decode(predicted_tokens[0])
print(predicted_text)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Alan Turing is a mathematician, mathematician, and mathematician.
Alan Turing is one of the world's greatest mathematicians


In [13]:
import torch
full_text = text
tokens = tokenizer.encode(text, return_tensors="pt")
for _ in range(8):
    logits = model(tokens).logits
    next_token = logits[:, -1, :].argmax(dim=-1).unsqueeze(0)
    tokens = torch.cat([tokens, next_token], dim=1)
print(tokenizer.decode(tokens[0], skip_special_tokens=True))

Alan Turing is a mathematician, mathematician, and mathematician.


In [14]:
import torch
import torch.nn.functional as F

tokens = tokenizer.encode(text, return_tensors="pt")

beam_size = 3
max_steps = 8

beams = [(tokens, 0.0)]

for _ in range(max_steps):

    new_beams = []

    for seq, score in beams:

        logits = model(seq).logits[:, -1, :]
        probs = F.log_softmax(logits, dim=-1)

        topk_probs, topk_ids = probs.topk(beam_size)

        for k in range(beam_size):

            next_token = topk_ids[:, k].unsqueeze(0)
            new_seq = torch.cat([seq, next_token], dim=1)

            new_score = score + topk_probs[0, k].item()

            new_beams.append((new_seq, new_score))

    beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

best_tokens = beams[0][0]

print(tokenizer.decode(best_tokens[0], skip_special_tokens=True))


Alan Turing is one of the world's greatest mathematicians


# Recurrent Neural Networks

https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html#torch.nn.RNN

input_size -> Girdi Boyutu      
hidden_size -> Context Vector boyutu        
num_layers -> Stack'lenen RNN sayısı. 1'den büyük olması Deep RNN olduğu anlamına gelir.        
nonlinearity -> "tanh" veya "relu" olarak seçebilirsiniz.       
bias -> bias eklenip eklenmeyeceği      
batch_first -> Eğer "True" ise -> (batch, seq, feature) , Eğer "False" ise -> (seq, batch, feature)     
dropout -> Eğer "True" ise output layer harici her bir RNN layer için dropout uygular. Sadece Deep RNN için kullanılabilir.     
bidirectional -> RNN'in bidirectional olup olmayacağı

In [16]:
import torch
import torch.nn as nn

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size) #Zero init

        out, _ = self.rnn(x, h0)

        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [19]:
input_size = 10
hidden_size = 32
num_layers = 1
num_classes = 5

model = RNNModel(input_size, hidden_size, num_layers, num_classes)

x = torch.randn(8, 30, 10)  # batch=8, seq_len=20, feature=10
y = model(x)

print(y.shape)  # (8,5)

torch.Size([8, 5])


# GRU

https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html

input_size -> Girdi Boyutu      
hidden_size -> Context Vector boyutu        
num_layers -> Stack'lenen GRU sayısı. 1'den büyük olması Deep GRU olduğu anlamına gelir.        
bias -> bias eklenip eklenmeyeceği      
batch_first -> Eğer "True" ise -> (batch, seq, feature) , Eğer "False" ise -> (seq, batch, feature)     
dropout -> Eğer "True" ise output layer harici her bir GRU layer için dropout uygular. Sadece Deep GRU için kullanılabilir.     
bidirectional -> GRU'in bidirectional olup olmayacağı

In [20]:
import torch
import torch.nn as nn

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, hn = self.gru(x, h0)

        out = out[:, -1, :]   # son timestep
        out = self.fc(out)

        return out


In [25]:
model = GRUModel(
    input_size=10,
    hidden_size=64,
    num_layers=2,
    num_classes=5
)

x = torch.randn(8, 20, 10)
y = model(x)
print(y.shape)  # (8,5)


torch.Size([8, 5])


# LSTM

https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html

input_size -> Girdi Boyutu      
hidden_size -> Context Vector boyutu        
num_layers -> Stack'lenen LSTM sayısı. 1'den büyük olması Deep LSTM olduğu anlamına gelir.             
bias -> bias eklenip eklenmeyeceği      
batch_first -> Eğer "True" ise -> (batch, seq, feature) , Eğer "False" ise -> (seq, batch, feature)     
dropout -> Eğer "True" ise output layer harici her bir LSTM layer için dropout uygular. Sadece Deep LSTM için kullanılabilir.     
bidirectional -> LSTM'in bidirectional olup olmayacağı

In [1]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, (hn, cn) = self.lstm(x, (h0, c0))

        out = out[:, -1, :]   # son timestep
        out = self.fc(out)

        return out

In [3]:
model = LSTMModel(
    input_size=10,
    hidden_size=64,
    num_layers=2,
    num_classes=5
)

x = torch.randn(8, 20, 10)
y = model(x)
print(y.shape)  # (8,5)

torch.Size([8, 5])


# Attention

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.V  = nn.Linear(hidden_dim, 1)

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: (batch, hidden_dim)
        encoder_outputs: (batch, seq_len, hidden_dim)
        """
        decoder_hidden = decoder_hidden.unsqueeze(1)

        score = self.V(
            torch.tanh(
                self.W1(encoder_outputs) +
                self.W2(decoder_hidden)
            )
        )

        attn_weights = F.softmax(score, dim=1)

        context = torch.sum(attn_weights * encoder_outputs, dim=1)

        return context, attn_weights

In [7]:
batch_size = 1
seq_len = 5
hidden_dim = 16

encoder_outputs = torch.randn(batch_size, seq_len, hidden_dim)
decoder_hidden  = torch.randn(batch_size, hidden_dim)

attention = Attention(hidden_dim)

context, weights = attention(decoder_hidden, encoder_outputs)

print("Context shape:", context.shape)
print("Attention weights shape:", weights.shape)
print(weights)

Context shape: torch.Size([1, 16])
Attention weights shape: torch.Size([1, 5, 1])
tensor([[[0.1703],
         [0.3119],
         [0.1803],
         [0.2202],
         [0.1174]]], grad_fn=<SoftmaxBackward0>)


# Self-Attention - MultiHead Attention

In [1]:
import torch
import torch.nn as nn

seq_len = 5

mha = nn.MultiheadAttention(embed_dim=300, 
                            num_heads=1, 
                            batch_first=True)

x = torch.rand(1, seq_len, 300) # (batch_size, seq_len, d_model)

out, attn_weights = mha(x, x, x)

print("Output shape:", out.shape)             # (batch_size, seq_len, d_model)
print("Attention weights shape:", attn_weights.shape)  # (batch_size, seq_len, seq_len)
print(attn_weights)

Output shape: torch.Size([1, 5, 300])
Attention weights shape: torch.Size([1, 5, 5])
tensor([[[0.1891, 0.1895, 0.2279, 0.2098, 0.1838],
         [0.1997, 0.1799, 0.2300, 0.2131, 0.1774],
         [0.1944, 0.1843, 0.2195, 0.2157, 0.1860],
         [0.1923, 0.1865, 0.2279, 0.2167, 0.1766],
         [0.1918, 0.1911, 0.2341, 0.2083, 0.1749]]], grad_fn=<MeanBackward1>)


In [2]:
seq_len = 5

mha = nn.MultiheadAttention(embed_dim=300, 
                            num_heads=2, 
                            batch_first=True)

x = torch.rand(1, seq_len, 300) # (batch_size, seq_len, d_model)

out, attn_weights = mha(x, x, x)

out, attn_weights = mha(x, x, x, average_attn_weights=False) # Her head ayrı attention döndür

print("Output shape:", out.shape)
print("Attention weights shape:", attn_weights.shape)  
print(attn_weights)

Output shape: torch.Size([1, 5, 300])
Attention weights shape: torch.Size([1, 2, 5, 5])
tensor([[[[0.1940, 0.1996, 0.2124, 0.1990, 0.1950],
          [0.2002, 0.1919, 0.2184, 0.1980, 0.1915],
          [0.2104, 0.1990, 0.1944, 0.2118, 0.1843],
          [0.1873, 0.1941, 0.2180, 0.2195, 0.1810],
          [0.2003, 0.1965, 0.2121, 0.2056, 0.1856]],

         [[0.1955, 0.2105, 0.2011, 0.1970, 0.1959],
          [0.1997, 0.2057, 0.1937, 0.1907, 0.2102],
          [0.1970, 0.1897, 0.2088, 0.2032, 0.2014],
          [0.1928, 0.2079, 0.1928, 0.2096, 0.1970],
          [0.2049, 0.1969, 0.1983, 0.2012, 0.1987]]]], grad_fn=<ViewBackward0>)


# Positional Encoding

In [3]:
import numpy as np

def positional_encoding(seq_len, d_model, max_len=512):
    """
    Sin/Cos tabanlı Positional Encoding
    """
    PE = np.zeros((max_len, d_model))
    positions = np.arange(max_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (dims//2)) / d_model)
    PE[:, 0::2] = np.sin(positions * angle_rates[:, 0::2])
    PE[:, 1::2] = np.cos(positions * angle_rates[:, 1::2])

    return PE[:seq_len, :]

def apply_pe_to_embeddings(embeddings):
    """
    embeddings: (seq_len, d_model) numpy array
    return: embeddings + positional_encoding
    """
    seq_len, d_model = embeddings.shape
    PE = positional_encoding(seq_len, d_model)
    return embeddings + PE

seq_len = 5
d_model = 8

word_embeddings = np.random.rand(seq_len, d_model)

embeddings_with_pe = apply_pe_to_embeddings(word_embeddings)

print("Word Embeddings:\n", word_embeddings)
print("\nPositional Encoding eklenmiş Embeddings:\n", embeddings_with_pe)


Word Embeddings:
 [[0.29350664 0.82365459 0.95397616 0.22834183 0.55213777 0.24372589
  0.17284943 0.05766212]
 [0.811474   0.52626182 0.81057946 0.96155052 0.07580429 0.39150546
  0.17542819 0.68609824]
 [0.24321562 0.90642019 0.76444666 0.35514797 0.72087862 0.59814512
  0.97996169 0.33066074]
 [0.47832237 0.80002865 0.13579394 0.52573001 0.78333684 0.8813668
  0.58760134 0.40372134]
 [0.06432595 0.18848053 0.88022851 0.42563261 0.86156241 0.04752915
  0.12199196 0.25388874]]

Positional Encoding eklenmiş Embeddings:
 [[ 0.29350664  1.82365459  0.95397616  1.22834183  0.55213777  1.24372589
   0.17284943  1.05766212]
 [ 1.65294499  1.06656412  0.91041288  1.95655468  0.08580412  1.39145546
   0.17642819  1.68609774]
 [ 1.15251304  0.49027335  0.96311599  1.33521455  0.74087729  1.59794512
   0.98196169  1.33065874]
 [ 0.61944238 -0.18996385  0.43131414  1.4810665   0.81333234  1.88091684
   0.59060134  1.40371684]
 [-0.69247654 -0.46516309  1.26964686  1.3466936   0.90155175  1.04672